# Interactive Hook Sessions (Offline)

`HookSession` is TDHook's imperative interface for temporary capture and intervention. This notebook presents its targets, result objects, lifecycle, workflow integration, and managed early stopping with a local model.

In [ ]:
import torch
from tensordict import TensorDict
from torch import nn

from tdhook.session import HookSession
from tdhook.targets import Target

torch.manual_seed(0)


class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(4, 6)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(6, 2)

    def forward(self, input):
        return self.linear2(self.relu(self.linear1(input)))


model = TinyModel().eval()
inputs = torch.randn(3, 4)
data = TensorDict({"input": inputs}, batch_size=[len(inputs)])

## Targets and capture results

A `Target` names a model-relative module, a value kind, its feature axis, and selected indices. `capture` returns a mutable `CapturedTarget`: `value` is the latest detached observation, while `values` retains observations from every matching call in order.

In [ ]:
activation = Target("linear1", "activation", -1, (0, 2))

with HookSession(model) as session:
    captured = session.capture(activation)
    first_output = model(inputs)
    second_output = model(inputs + 1)
    active_program = session.program

assert captured.value is captured.values[-1]
assert len(captured.values) == 2
assert captured.values[0].shape == (3, 2)
assert len(active_program.hooks) == 1
[tuple(value.shape) for value in captured.values]

## Activation and parameter replacement

`replace` applies only while the context is active. Activation hooks are removed and parameter values are restored on exit, including exceptional exits.

In [ ]:
baseline = model(inputs)
output_unit = Target("", "activation", -1, (0,))
parameter_row = Target("linear1", "parameter", 0, (0,), parameter="weight")
original_weight = model.linear1.weight.detach().clone()

with HookSession(model) as session:
    session.replace(output_unit, 0)
    session.replace(parameter_row, -1)
    changed = model(inputs)
    assert torch.equal(changed[:, 0], torch.zeros_like(changed[:, 0]))
    assert not torch.equal(model.linear1.weight, original_weight)

assert torch.equal(model.linear1.weight, original_weight)
torch.testing.assert_close(model(inputs), baseline)

## Forward input operations

Pass `direction="fwd_pre"` to capture or replace positional forward inputs. `direction="fwd_pre_kwargs"` exposes the hook value as `(args, kwargs)`; use `output_path=(0, 0)` for the first positional argument or `output_path=(1, "scale")` for keyword `scale`. The selected tuple, list, mapping, or TensorDict structure is preserved.

In [ ]:
input_unit = Target("linear1", "activation", -1, (0,), output_path=(0,))

with HookSession(model) as session:
    captured_input = session.capture(input_unit, direction="fwd_pre")
    session.replace(input_unit, -1, direction="fwd_pre")
    input_modified_output = model(inputs)

torch.testing.assert_close(captured_input.value, inputs[:, :1])
assert input_modified_output.shape == (3, 2)

## Gradient operations

Gradient targets use the same capture and replacement interface. `direction="bwd"` selects gradient inputs and `direction="bwd_pre"` selects gradient outputs; `bwd_pre` remains the default for compatibility. Backward must run inside the session so its temporary backward hooks are still installed.

In [ ]:
gradient_input = inputs.detach().clone().requires_grad_()
gradient_output = Target("linear1", "gradient", -1, (0,))
gradient_input_target = Target("linear1", "gradient", -1, (0,))

with HookSession(model) as session:
    captured_gradient_output = session.capture(gradient_output)
    captured_gradient_input = session.capture(gradient_input_target, direction="bwd")
    session.replace(gradient_input_target, 0, direction="bwd")
    model(gradient_input).sum().backward()

assert captured_gradient_output.value is not None
assert captured_gradient_output.value.shape == (3, 1)
assert captured_gradient_input.value is not None
assert captured_gradient_input.value.shape == (3, 1)
assert gradient_input.grad is not None
assert torch.equal(gradient_input.grad[:, 0], torch.zeros(3))

## Wrap a declared workflow

`workflow.session(model)` exposes the same operations around the complete workflow run. Session operations are reported beside the exact executed plan, but remain outside workflow planning and co-execution decisions.

In [ ]:
from tdhook.latent import ActivationCaching
from tdhook.workflow import Workflow

workflow = Workflow(
    ActivationCaching("linear1", cache_key=("activations", "first")),
    ActivationCaching("linear2", cache_key=("activations", "second")),
)

with workflow.session(model) as session:
    workflow_capture = session.capture(activation)
    execution = session(data.clone())

assert execution.plan.model_passes == 1
assert execution.program == session.program
assert len(workflow_capture.values) == 1
execution.plan

## Managed early stopping

`stop` ends forward execution after a selected module and suppresses TDHook's internal control-flow signal at the session boundary. It returns an `EarlyStopResult` containing the exact partial output. In a workflow session, stopping aborts the complete workflow, so no `WorkflowResult` is produced. Gradient operations cannot be combined with early stopping.

In [ ]:
with workflow.session(model) as session:
    stopped = session.stop("relu")
    session(data.clone())

assert stopped.reached
assert stopped.output is not None
assert stopped.output.shape == (3, 6)
assert session.program.stopped_at == "relu"
assert all(not module._forward_hooks for module in model.modules())
stopped.output

## Choosing an interface

Use `method.prepare(model)` for one configured method, `Workflow` for declared composition, and `HookSession` for interactive operations whose lifecycle you control directly. Use `workflow.session(model)` only when those imperative operations must wrap every model execution chosen by a workflow plan.